# 03 — Training
PyTorch Lightning: `Trainer().fit()` — không dùng vòng lặp `for` thủ công.

**Cấu hình theo Paper:**
- Optimizer: SGD (lr=0.001, momentum=0.9, weight_decay=0.001)
- Scheduler: ReduceLROnPlateau (patience=10, factor=0.1, min_lr=1e-6)
- Max epochs: 150 | Batch: 30 | Early stopping: 15 epochs
- 5-Fold Stratified CV
- **Progress bar per epoch: train_loss / val_loss / val_bcc hiển thị trực tiếp**

In [ ]:
import sys, os
sys.path.insert(0, '..')

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
import torch.nn.functional as F

import pytorch_lightning as pl
from pytorch_lightning.callbacks import (
    EarlyStopping, ModelCheckpoint, TQDMProgressBar
)
from pytorch_lightning.loggers import CSVLogger

from skin_pipeline.utils import (
    SkinLesionDataset, SkinLesionModel,
    get_transforms, CLASSES, NUM_CLASSES
)
import warnings; warnings.filterwarnings('ignore')

# ── EpochSummaryCallback: scrollable Keras-style epoch history ────────────
import sys as _sys
class EpochSummaryCallback(pl.Callback):
    def __init__(self):
        self._best = float('inf')

    def on_validation_epoch_end(self, trainer, pl_module):
        if trainer.sanity_checking:
            return
        m   = trainer.callback_metrics
        ep  = trainer.current_epoch + 1
        mx  = trainer.max_epochs
        tl  = float(m.get('train_loss', float('nan')))
        vl  = float(m.get('val_loss',   float('nan')))
        bcc = float(m.get('val_bcc',    float('nan')))
        lr  = trainer.optimizers[0].param_groups[0]['lr']
        tag = '  ← best' if vl < self._best else ''
        if vl < self._best:
            self._best = vl
        print(f'\nEpoch {ep:>3}/{mx} — '
              f'train_loss: {tl:.4f}  val_loss: {vl:.4f}  '
              f'val_bcc: {bcc:.4f}  lr: {lr:.2e}{tag}')
        _sys.stdout.flush()


IndentationError: unexpected indent (265097615.py, line 28)

## 1. Config

In [ ]:
DATA_DIR = os.path.expanduser('~/.cache/kagglehub/datasets/mahdavi1202/skin-cancer/versions/1')
CSV_PATH = 'processed_metadata.csv'          
CKPT_DIR = 'checkpoints'
LOG_DIR  = 'logs'


# Đổi BACKBONE và USE_METABLOCK để chuyển đổi giữa 2 model:
#   PiT + MetaBlock  
#   ResNetBit + Concat  
BACKBONE      = 'pit_s_distilled_224'        # hoặc 'resnetv2_50x1_bit_distilled'
USE_METABLOCK = True                         # False = concat baseline
IMG_SIZE      = 224
BATCH_SIZE    = 30
MAX_EPOCHS    = 150
N_FOLDS       = 5
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'Model: {BACKBONE} | MetaBlock: {USE_METABLOCK}')

## 2. LightningDataModule

In [ ]:
class SkinDataModule(pl.LightningDataModule):
    def __init__(self, data_dir, csv_path, batch_size, img_size, fold):
        super().__init__()
        self.data_dir   = data_dir
        self.csv_path   = csv_path
        self.batch_size = batch_size
        self.img_size   = img_size
        self.fold       = fold

    def setup(self, stage=None):
        df = pd.read_csv(self.csv_path)
        self.feature_cols = [c for c in df.columns if c not in
                             {'patient_id','lesion_id','diagnostic','diagnostic_idx','img_id'}]
        self.meta_dim = len(self.feature_cols)

        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
        train_idx, val_idx = list(skf.split(df, df['diagnostic_idx']))[self.fold]

        self.train_ds = SkinLesionDataset(
            self.data_dir, df.iloc[train_idx].copy(), self.feature_cols,
            transform=get_transforms(self.img_size, train=True))
        self.val_ds = SkinLesionDataset(
            self.data_dir, df.iloc[val_idx].copy(), self.feature_cols,
            transform=get_transforms(self.img_size, train=False))

        
        y_train = self.train_ds.labels
        classes, counts = np.unique(y_train, return_counts=True)
        cw_dict = {int(c): len(y_train) / (len(classes) * cnt) for c, cnt in zip(classes, counts)}
        weights = [cw_dict.get(c, 1.0) for c in range(NUM_CLASSES)]
        self.class_weights = torch.FloatTensor(weights)
        print(f'Fold {self.fold} Class Weights:\n', [round(w, 2) for w in weights])

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True,  num_workers=2)
    def val_dataloader(self):
        return DataLoader(self.val_ds,   batch_size=self.batch_size, shuffle=False, num_workers=2)


## 3. LightningModule

In [ ]:
class SkinLightningModel(pl.LightningModule):
    def __init__(self, backbone, meta_dim, use_metablock=True, class_weights=None):
        super().__init__()
        self.save_hyperparameters()
        self.model = SkinLesionModel(backbone, meta_dim, use_metablock=use_metablock)
        
        # DÙNG CLASS WEIGHTS
        self.criterion = nn.CrossEntropyLoss(weight=class_weights)
        
        self._val_preds, self._val_labels = [], []

    def forward(self, img, meta):
        return self.model(img, meta)

    def training_step(self, batch, _):
        img, meta, y = batch
        loss = self.criterion(self(img, meta), y)
        self.log('train_loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, _):
        img, meta, y = batch
        logits = self(img, meta)
        loss   = self.criterion(logits, y)
        preds  = torch.argmax(logits, dim=1)
        self._val_preds.append(preds.cpu())
        self._val_labels.append(y.cpu())
        self.log('val_loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def on_validation_epoch_end(self):
        preds  = torch.cat(self._val_preds).numpy()
        labels = torch.cat(self._val_labels).numpy()
        bcc    = balanced_accuracy_score(labels, preds)
        self.log('val_bcc', bcc, prog_bar=True)
        self._val_preds.clear(); self._val_labels.clear()

    def configure_optimizers(self):
        opt = optim.SGD(self.parameters(), lr=0.001, momentum=0.9, weight_decay=0.001)
        sch = {
            'scheduler': torch.optim.lr_scheduler.ReduceLROnPlateau(
                opt, mode='min', patience=10, factor=0.1, min_lr=1e-6),
            'monitor': 'val_loss',
        }
        return [opt], [sch]


## 4. Train

In [ ]:
for FOLD in range(N_FOLDS):
    print(f'\n======================================================')
    print(f'         BẮT ĐẦU HUẤN LUYỆN FOLD {FOLD} / {N_FOLDS-1}')
    print(f'======================================================')
    
    # ── Data ─────────────────────────────────────────────────────────────────
    dm = SkinDataModule(DATA_DIR, CSV_PATH, BATCH_SIZE, IMG_SIZE, fold=FOLD)
    dm.setup()
    print(f'Fold {FOLD} | Train: {len(dm.train_ds)} | Val: {len(dm.val_ds)} | Features: {dm.meta_dim}')
    
    # ── Model 
    lit_model = SkinLightningModel(
        BACKBONE, dm.meta_dim, 
        use_metablock=USE_METABLOCK,
        class_weights=dm.class_weights
    )
    
    # ── Callbacks 
    early_stop = EarlyStopping(monitor='val_loss', patience=15, mode='min', verbose=True)
    checkpoint = ModelCheckpoint(
        dirpath=f'{CKPT_DIR}/fold{FOLD}',
        filename=f'{BACKBONE.replace("/","-")}-fold{FOLD}-{{epoch:02d}}-{{val_bcc:.3f}}',
        monitor='val_bcc', mode='max', save_top_k=1
    )
    progress_bar = TQDMProgressBar(refresh_rate=1)
    epoch_summary = EpochSummaryCallback()
    
    # ── Logger ───
    logger = CSVLogger(LOG_DIR, name=f'{BACKBONE}_fold{FOLD}')
    
    # ── Trainer ──
    trainer = pl.Trainer(
        max_epochs=MAX_EPOCHS,
        accelerator='auto',
        callbacks=[early_stop, checkpoint, progress_bar, epoch_summary],
        logger=logger,
        log_every_n_steps=1,
        enable_progress_bar=True,
        gradient_clip_val=1.0,  # Bắt buộc: chặn bùng nổ đạo hàm gây model collapse
    )
    
    trainer.fit(lit_model, dm)


## 5. Plot Metrics Sau Khi Train

In [ ]:
import glob

for FOLD in range(N_FOLDS):
    log_files = sorted(glob.glob(f'{LOG_DIR}/{BACKBONE}_fold{FOLD}/version_*/metrics.csv'))
    if log_files:
        metrics = pd.read_csv(log_files[-1])
        # Gộp train/val theo epoch
        epoch_df = metrics.groupby('epoch').last().reset_index()

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Loss
        axes[0].plot(epoch_df['epoch'], epoch_df['train_loss'], label='Train Loss', marker='.')
        axes[0].plot(epoch_df['epoch'], epoch_df['val_loss'],   label='Val Loss',   marker='.')
        axes[0].set_title(f'Fold {FOLD} — Loss per Epoch')
        axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('CrossEntropy Loss')
        axes[0].legend(); axes[0].grid(True)

        # BCC
        axes[1].plot(epoch_df['epoch'], epoch_df['val_bcc'], color='green', label='Val BCC', marker='s')
        axes[1].set_title(f'Fold {FOLD} — Balanced Accuracy per Epoch')
        axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('BCC')
        axes[1].legend(); axes[1].grid(True)

        plt.suptitle(f'{BACKBONE} | MetaBlock={USE_METABLOCK} | Fold {FOLD}', fontsize=13)
        plt.tight_layout(); plt.show()
        print(f"Best Val BCC for Fold {FOLD}: {epoch_df['val_bcc'].max():.4f}\n")
    else:
        print(f'Log file not found for Fold {FOLD}.')
